# Step 20 — Does the census belong to the data or to the classifier?

Responds to review points 1, 2, 3, 7 and the minor points.

**A. An exactly symmetric null that goes through a classifier.** One 30-class noise-conditional classifier is trained on the 10 CIFAR-10 classes, their vertical flips, and their red–blue channel swaps. Both maps are orthogonal and commute with isotropic Gaussian noise, so each pair $(c, g\cdot c)$ is exactly symmetric in pixel space. For the Bayes coordinate $h=0$ exactly, so the measured $|h|$ is the classifier plus the estimator. We also symmetrise the coordinate, $z_{\rm sym}(x)=\tfrac12[z(x)-z(gx)]$, which isolates the estimator. Horizontal flips are not used as a null, because the classifier is trained with horizontal-flip augmentation.

**B. Is the coordinate an affine function of a Bayes log-LR?** A coordinate $u$ is affine in the Bayes log-likelihood ratio if and only if $\log p(u\mid a)/p(u\mid b)$ is linear in $u$. Our pipeline is invariant to affine maps. So, per pair and noise level, we fit a spline-logistic calibration of $z$ on half of the test images, and measure its nonlinearity and slope. We then redo the census on the other half along the calibrated coordinate, and along two deliberately non-canonical warps ($\tanh(z/2)$ and signed $\sqrt{|z|}$).

**C. A second architecture.** A plain VGG-style network with GroupNorm and a different noise conditioning (an input-scaling preconditioner plus a $\log\sigma$ channel), trained on the same data.

**D. A Gaussian baseline.** Pixel-space class means and pooled covariance predict $\sigma_G$, where the discriminant separation falls to 2. How much of $\sigma_{\rm bif}$ and of the hierarchy is second-order statistics?

**E. Statistics.**
- 1,000 bootstrap resamples, and effect sizes with the equivalent class imbalance $e^{2\,\mathrm{artanh}\,h}$.
- Class-level permutation and leave-one-class-out tests for the tighter-class rule and for the hierarchy.
- Sensitivity to bandwidth, merge thresholds and a persistence rule.
- Test-set accuracy of the classifier at each $\sigma$.

Logits are computed once on a 56-level noise grid and every analysis reads them, so all pairs and variants are cheap.

**Runtime.** Self-contained. GPU on, Internet on. Run as a committed version, about 3–4 h. If a Step 16/18/19 output is mounted, the paper's 10-class classifier is loaded instead of retrained.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
from scipy.stats import spearmanr
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); plt.rcParams['figure.dpi'] = 120
rng = np.random.default_rng(0); torch.manual_seed(0)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True); dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True)
XTR = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YTR = np.asarray(ds.targets)
XTE = torch.from_numpy((np.asarray(dst.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YTE = np.asarray(dst.targets)
VFLIP = lambda x: x.flip(-2); SWAP = lambda x: x[:, [2, 1, 0]]          # orthogonal pixel maps that commute with isotropic noise
SIG = np.geomspace(20.0, 0.08, 56)                                       # descending noise grid (~10.5% steps)
print('noise grid', SIG[0], '...', SIG[-1])

# ---------------- classifiers ----------------
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    """The paper's classifier (ResNet with FiLM noise conditioning)."""
    def __init__(self, n_out=10, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, n_out)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
class NoisyVGG(nn.Module):
    """A deliberately different family: plain VGG stack (no residuals), GroupNorm, input preconditioning x/sqrt(1+sigma^2),
    noise level entering as an extra constant input channel (no FiLM)."""
    def __init__(self, n_out=10, w=64):
        super().__init__(); L = []; cin = 4
        for cout, pool in [(w, False), (w, True), (2*w, False), (2*w, True), (4*w, False), (4*w, True)]:
            L += [nn.Conv2d(cin, cout, 3, 1, 1), nn.GroupNorm(8, cout), nn.GELU()] + ([nn.MaxPool2d(2)] if pool else []); cin = cout
        self.body = nn.Sequential(*L); self.fc = nn.Linear(cin, n_out)
    def forward(self, x, sigma):
        s = sigma.view(-1, 1, 1, 1); x = x/torch.sqrt(1 + s**2); c = (0.25*torch.log(s)).expand(-1, 1, x.shape[2], x.shape[3])
        return self.fc(self.body(torch.cat([x, c], 1)).mean((2, 3)))
def train_clf(model, X, Y, seed, epochs=25, bs=256, s_lo=0.05, s_hi=10.0, tag=''):
    torch.manual_seed(seed); Xd = X.to(dev); Yd = torch.from_numpy(Y).long().to(dev); opt = torch.optim.AdamW(model.parameters(), 2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs*(len(Xd)//bs)); t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xd), device=dev)
        for i in range(0, len(Xd)-bs+1, bs):
            j = perm[i:i+bs]; x0 = Xd[j]
            if torch.rand(1).item() < 0.5: x0 = x0.flip(-1)                       # horizontal flip commutes with VFLIP and SWAP
            sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(s_lo), math.log(s_hi))); xt = x0 + sigma.view(-1, 1, 1, 1)*torch.randn_like(x0)
            with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')): loss = F.cross_entropy(model(xt, sigma), Yd[j])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval(); del Xd; torch.cuda.empty_cache(); print(f"trained {tag} ({time.time()-t0:.0f}s)"); return model
@torch.no_grad()
def logits_grid(model, X, repeats, seed, transform=None, bs=1000):
    """Logits of model at every noise level of SIG for X (optionally transformed AFTER noising), R noise draws per image. Returns float16 [len(SIG), R*len(X), C]."""
    g = torch.Generator(device=dev).manual_seed(seed); C_ = None; out = None
    for r in range(repeats):
        for i0 in range(0, len(X), bs):
            x0 = X[i0:i0+bs].to(dev); eps = torch.randn(x0.shape, generator=g, device=dev)
            for gi, s in enumerate(SIG):
                xt = x0 + float(s)*eps
                if transform is not None: xt = transform(xt)
                with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')): l = model(xt, torch.full((len(xt),), float(s), device=dev)).float()
                if out is None: C_ = l.shape[1]; out = np.zeros((len(SIG), repeats*len(X), C_), np.float16)
                out[gi, r*len(X)+i0: r*len(X)+i0+len(x0)] = l.cpu().numpy().astype(np.float16)
    return out

In [ ]:
# ---- the paper's 10-class classifier (loaded if a previous run is mounted), the 30-class null classifier, and the second architecture ----
found = glob.glob('/kaggle/input/**/noisy_resnet_seed0.pt', recursive=True)
if found: CLF = NoisyResNet(10).to(dev); CLF.load_state_dict(torch.load(found[0], map_location=dev)); CLF.eval(); print('loaded paper classifier from', found[0])
else: CLF = train_clf(NoisyResNet(10).to(dev), XTR, YTR, 0, tag='paper ResNet (10 classes)'); torch.save(CLF.state_dict(), '/kaggle/working/noisy_resnet_seed0.pt')
X30 = torch.cat([XTR, VFLIP(XTR), SWAP(XTR)]); Y30 = np.concatenate([YTR, YTR + 10, YTR + 20])
CLF30 = train_clf(NoisyResNet(30).to(dev), X30, Y30, 0, tag='null ResNet (30 classes: real, vflip, R-B swap)'); torch.save(CLF30.state_dict(), '/kaggle/working/noisy_resnet30.pt'); del X30
VGG = train_clf(NoisyVGG(10).to(dev), XTR, YTR, 0, tag='VGG (10 classes)'); torch.save(VGG.state_dict(), '/kaggle/working/noisy_vgg.pt')
# test accuracy by noise level
@torch.no_grad()
def test_acc(model, n_cls=10, sigmas=(0.05, 0.25, 0.5, 1.0, 2.0, 4.0)):
    out = {}
    for s in sigmas:
        corr = 0
        for i0 in range(0, len(XTE), 1000):
            x = XTE[i0:i0+1000].to(dev); x = x + s*torch.randn_like(x)
            with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')): l = model(x, torch.full((len(x),), s, device=dev)).float()[:, :10]
            corr += int((l.argmax(1).cpu().numpy() == YTE[i0:i0+1000]).sum())
        out[s] = corr/len(XTE)
    return out
for nm, m in [('paper ResNet', CLF), ('30-class ResNet (real logits)', CLF30), ('VGG', VGG)]: print(f"TEST accuracy {nm:>30}: " + ', '.join(f"sigma {k}: {v:.3f}" for k, v in test_acc(m).items()))
t0 = time.time()
L_MAIN = logits_grid(CLF, XTE, 4, 11); print('paper classifier, test x4:', L_MAIN.shape, f'{time.time()-t0:.0f}s')
L_VGG = logits_grid(VGG, XTE, 4, 11); print('VGG, test x4:', L_VGG.shape, f'{time.time()-t0:.0f}s')
# 30-class: original test images, their vflips, their swaps (2 noise draws each); plus logits of the transformed noisy images for the symmetrised null
L30_O = logits_grid(CLF30, XTE, 2, 21); L30_V = logits_grid(CLF30, VFLIP(XTE), 2, 22); L30_S = logits_grid(CLF30, SWAP(XTE), 2, 23)
L30_Og = logits_grid(CLF30, XTE, 2, 21, transform=VFLIP); L30_Vg = logits_grid(CLF30, VFLIP(XTE), 2, 22, transform=VFLIP)
L30_Os = logits_grid(CLF30, XTE, 2, 21, transform=SWAP);  L30_Ss = logits_grid(CLF30, SWAP(XTE), 2, 23, transform=SWAP)
print(f'30-class logits done ({time.time()-t0:.0f}s)')
YT4 = np.tile(YTE, 4); YT2 = np.tile(YTE, 2); IMG4 = np.tile(np.arange(len(YTE)), 4)
np.savez_compressed('/kaggle/working/step20_logits.npz', L_MAIN=L_MAIN, L_VGG=L_VGG, SIG=SIG, YTE=YTE)

## Shared census machinery (grid scan on stored logits)

In [ ]:
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def kde_score(m, bwf=1.5):
    m = np.clip(m, -8, 8); bw = max(bwf*1.06*m.std()*len(m)**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, merge=0.03, prominence=0.05, lo=-3.5, hi=3.5):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx], merge)
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_fit(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def standardize(z, ya):
    mua, mub = z[ya].mean(), z[~ya].mean(); half = 0.5*(mua-mub); sep = abs(half)/(0.5*(z[ya].std()+z[~ya].std())); return (z - 0.5*(mua+mub))/half, sep
def census_from_z(Zg, ya, bwf=1.5, merge=0.03, prominence=0.05, persist=1, min_sep=0.3):
    """Zg: list over the (descending) noise grid of raw coordinate values for the pair's samples; ya: boolean 'is class a'.
    Returns dict(sigma, gi, h, a, b, clamped) at the first bimodal grid point (bimodality must persist for `persist` consecutive points)."""
    seen_uni = False; run = 0
    for gi in range(len(SIG)):
        m, sep = standardize(Zg[gi], ya)
        if sep < min_sep: seen_uni = True; run = 0; continue
        sm, p = kde_score(m, bwf); nm = n_max(stationary(sm, p, merge, prominence))
        if nm >= 2 and seen_uni:
            run += 1
            if run >= persist:
                g0 = gi - persist + 1; m0, _ = standardize(Zg[g0], ya); sm0, p0 = kde_score(m0, bwf); return dict(sigma=float(SIG[g0]), gi=g0, **cusp_fit(sm0, p0))
        elif nm == 1: seen_uni = True; run = 0
    return None
def pair_rows(Y, a, b): return np.where((Y == a) | (Y == b))[0]
def z_pair(L, rows, ca, cb): return [L[gi, rows, ca].astype(np.float64) - L[gi, rows, cb].astype(np.float64) for gi in range(len(SIG))]
def eq_imbalance(h): hh = np.clip(np.abs(h), 0, 0.95); r = np.exp(2*np.arctanh(hh)); return r/(1+r)      # h ~ tanh(eta) near the transition; weight share of the favoured class
print('ok')

## A. The symmetric null through the classifier

In [ ]:
def null_census(L_orig, L_trans, L_orig_g, L_trans_g, off, tag):
    """Null pair (c, g.c): samples = original test images of class c (a-side) and g-transformed test images of class c (b-side), both through CLF30.
    Raw coordinate: z = l_c - l_{c+off}. Symmetrised: z_sym(x) = (z(x) - z(g x))/2, using the logits of the transformed noisy image."""
    raw, sym = [], []
    for c in range(10):
        ra = np.where(YT2 == c)[0]; ya = np.concatenate([np.ones(len(ra), bool), np.zeros(len(ra), bool)])
        Zr = [np.concatenate([L_orig[gi, ra, c].astype(np.float64) - L_orig[gi, ra, c+off], L_trans[gi, ra, c].astype(np.float64) - L_trans[gi, ra, c+off]]) for gi in range(len(SIG))]
        Zs = [0.5*(Zr[gi] - np.concatenate([L_orig_g[gi, ra, c].astype(np.float64) - L_orig_g[gi, ra, c+off], L_trans_g[gi, ra, c].astype(np.float64) - L_trans_g[gi, ra, c+off]])) for gi in range(len(SIG))]
        r1 = census_from_z(Zr, ya); r2 = census_from_z(Zs, ya)
        raw.append(r1); sym.append(r2)
        print(f"  {tag} null {CLASSES[c]:>10}: raw h {r1['h'] if r1 else float('nan'):+.4f} (sigma {r1['sigma'] if r1 else float('nan'):.2f});  symmetrised h {r2['h'] if r2 else float('nan'):+.4f}")
    return raw, sym
NV_raw, NV_sym = null_census(L30_O, L30_V, L30_Og, L30_Vg, 10, 'vflip')
NS_raw, NS_sym = null_census(L30_O, L30_S, L30_Os, L30_Ss, 20, 'R-B swap')
null_raw = np.array([abs(r['h']) for r in NV_raw + NS_raw if r]); null_sym = np.array([abs(r['h']) for r in NV_sym + NS_sym if r])
# real pairs through the SAME 30-class classifier (original images only)
real30 = []
for a, b in PAIRS:
    rows = pair_rows(YT2, a, b); r = census_from_z(z_pair(L30_O, rows, a, b), YT2[rows] == a); real30.append(r)
h30 = np.array([abs(r['h']) for r in real30 if r])
FLOOR_NULL = float(np.percentile(null_raw, 95)) if len(null_raw) else np.nan
print(f"\nclassifier-induced null (20 exactly symmetric pairs through the classifier): |h| median {np.median(null_raw):.4f}, max {null_raw.max():.4f}")
print(f"estimator-only null (symmetrised coordinate): |h| median {np.median(null_sym):.4f}, max {null_sym.max():.4f}")
print(f"real pairs through the same 30-class classifier: |h| median {np.median(h30):.4f}; above the classifier null's MAX ({null_raw.max():.4f}): {(h30 > null_raw.max()).sum()}/{len(h30)}; above its 95th pct: {(h30 > FLOOR_NULL).sum()}/{len(h30)}")
plt.figure(figsize=(5, 3.2)); bins = np.linspace(0, max(h30.max(), null_raw.max())*1.05, 30)
plt.hist(h30, bins=bins, alpha=0.6, label='real pairs'); plt.hist(null_raw, bins=bins, alpha=0.7, label='symmetric pairs (a, g a) through the classifier'); plt.hist(null_sym, bins=bins, alpha=0.7, label='same, symmetrised coordinate')
plt.xlabel('|h|'); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

## B. Is the coordinate affine in a Bayes log-LR? Calibration, recalibrated census, and non-canonical warps

In [ ]:
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import LogisticRegression
HALF_A = (IMG4 % 2 == 0); HALF_B = ~HALF_A
def calibrate(z_fit, y_fit, z_eval):
    """Monotone-checked spline-logistic recalibration of z on held-out labels. Returns (u_eval, slope_linear, nonlinearity, monotone_fraction)."""
    mu, sd = z_fit.mean(), z_fit.std() + 1e-9; zf = ((z_fit-mu)/sd)[:, None]; ze = ((z_eval-mu)/sd)[:, None]
    lin = LogisticRegression(C=100.0, max_iter=2000).fit(zf, y_fit); slope = float(lin.coef_[0, 0]/sd)
    st = SplineTransformer(n_knots=6, degree=3, extrapolation='linear').fit(zf); spl = LogisticRegression(C=10.0, max_iter=4000).fit(st.transform(zf), y_fit)
    u = spl.decision_function(st.transform(ze)); ul = lin.decision_function(ze); nl = float(np.std(u - ul)/max(np.std(ul), 1e-9))
    q = np.linspace(np.percentile(ze, 1), np.percentile(ze, 99), 400)[:, None]; mono = float(np.mean(np.diff(spl.decision_function(st.transform(q))) > 0))
    return u, slope, nl, mono
def census_variants(L, a, b, rows_all):
    ya_all = YT4[rows_all] == a; rA = rows_all[HALF_A[rows_all]]; rB = rows_all[HALF_B[rows_all]]; yA = YT4[rA] == a; yB = YT4[rB] == a
    Zraw_B = z_pair(L, rB, a, b); Zraw_A = z_pair(L, rA, a, b)
    Zcal, stats = [], []
    for gi in range(len(SIG)):
        u, slope, nl, mono = calibrate(Zraw_A[gi], yA, Zraw_B[gi]); Zcal.append(u); stats.append((slope, nl, mono))
    out = dict(raw=census_from_z(Zraw_B, yB), cal=census_from_z(Zcal, yB), tanh=census_from_z([np.tanh(z/2) for z in Zraw_B], yB),
               sqrt=census_from_z([np.sign(z)*np.sqrt(np.abs(z)) for z in Zraw_B], yB))
    gi = out['raw']['gi'] if out['raw'] else len(SIG)//2; out['calib_at_bif'] = stats[gi]; return out
t0 = time.time(); CV = {}
for a, b in PAIRS:
    CV[(a, b)] = census_variants(L_MAIN, a, b, pair_rows(YT4, a, b))
print(f"({time.time()-t0:.0f}s)")
sl = np.array([v['calib_at_bif'][0] for v in CV.values()]); nl = np.array([v['calib_at_bif'][1] for v in CV.values()]); mo = np.array([v['calib_at_bif'][2] for v in CV.values()])
print(f"at each pair's transition: calibration slope of raw log-odds median {np.median(sl):.2f} (IQR {np.percentile(sl,25):.2f}-{np.percentile(sl,75):.2f}); "
      f"spline nonlinearity (sd of spline - linear, relative) median {np.median(nl):.3f}, max {nl.max():.3f}; monotone fraction min {mo.min():.2f}")
def compare(k):
    ok = [p for p in PAIRS if CV[p]['raw'] and CV[p][k]]; h0 = np.array([CV[p]['raw']['h'] for p in ok]); h1 = np.array([CV[p][k]['h'] for p in ok]); s0 = np.array([CV[p]['raw']['sigma'] for p in ok]); s1 = np.array([CV[p][k]['sigma'] for p in ok])
    return len(ok), np.corrcoef(h0, h1)[0, 1], np.median(np.abs(h1-h0)), np.corrcoef(np.log(s0), np.log(s1))[0, 1], np.median(s1/s0), np.mean(np.sign(h0) == np.sign(h1))
for k, lab in [('cal', 'calibrated log-LR (canonical)'), ('tanh', 'tanh(z/2) warp (non-canonical)'), ('sqrt', 'signed sqrt warp (non-canonical)')]:
    n, rh, dh, rs, ms, sg = compare(k); print(f"raw vs {lab:>33}: n={n}, corr h {rh:.3f}, median |dh| {dh:.4f}, sign agreement {sg:.2f}; corr log sigma {rs:.3f}, median ratio {ms:.2f}")

## C. Second architecture, and D. Gaussian baseline

In [ ]:
MAIN, VGGC = {}, {}
for a, b in PAIRS:
    rows = pair_rows(YT4, a, b); ya = YT4[rows] == a
    MAIN[(a, b)] = census_from_z(z_pair(L_MAIN, rows, a, b), ya); VGGC[(a, b)] = census_from_z(z_pair(L_VGG, rows, a, b), ya)
ok = [p for p in PAIRS if MAIN[p] and VGGC[p]]
hm = np.array([MAIN[p]['h'] for p in ok]); hv = np.array([VGGC[p]['h'] for p in ok]); sm_ = np.array([MAIN[p]['sigma'] for p in ok]); sv = np.array([VGGC[p]['sigma'] for p in ok])
print(f"ResNet vs VGG (different architecture family), {len(ok)} pairs bimodal under both: corr h {np.corrcoef(hm, hv)[0,1]:.3f}, sign agreement {np.mean(np.sign(hm)==np.sign(hv)):.2f}; corr log sigma {np.corrcoef(np.log(sm_), np.log(sv))[0,1]:.3f}, median ratio {np.median(sv/sm_):.2f}")
kind = lambda p: 'va' if ((p[0] in VEH) != (p[1] in VEH)) else ('vv' if p[0] in VEH else 'aa')
for k in ['va', 'vv', 'aa']:
    j = [i for i, p in enumerate(ok) if kind(p) == k]; print(f"   {k}: median sigma_bif ResNet {np.median(sm_[j]):.2f}, VGG {np.median(sv[j]):.2f}")
# Gaussian baseline: discriminant separation d'(sigma)^2 = sum_k (e_k.dmu)^2/(lam_k + sigma^2) with pooled pixel covariance; sigma_G where d' = 2
Xf = XTR.reshape(len(XTR), -1).to(dev).double(); SG = {}
for a, b in PAIRS:
    A = Xf[torch.from_numpy(YTR == a).to(dev)]; B = Xf[torch.from_numpy(YTR == b).to(dev)]; dmu = A.mean(0) - B.mean(0)
    S = 0.5*(torch.cov(A.T) + torch.cov(B.T)); lam, E = torch.linalg.eigh(S); pr = ((E.T @ dmu)**2).cpu().numpy(); lam = lam.clamp_min(0).cpu().numpy()
    dp = lambda s: np.sqrt(np.sum(pr/(lam + s**2))); lo, hi = 1e-3, 100.0
    if dp(lo) < 2: SG[(a, b)] = np.nan; continue
    for _ in range(60):
        mid = np.sqrt(lo*hi); lo, hi = (mid, hi) if dp(mid) > 2 else (lo, mid)
    SG[(a, b)] = np.sqrt(lo*hi)
del Xf; torch.cuda.empty_cache()
okg = [p for p in ok if np.isfinite(SG[p])]; sg = np.array([SG[p] for p in okg]); sb = np.array([MAIN[p]['sigma'] for p in okg])
print(f"Gaussian (second-order) baseline sigma_G: corr(log sigma_G, log sigma_bif) = {np.corrcoef(np.log(sg), np.log(sb))[0,1]:.3f}; median sigma_bif/sigma_G = {np.median(sb/sg):.2f}")
for k in ['va', 'vv', 'aa']:
    j = [i for i, p in enumerate(okg) if kind(p) == k]; print(f"   {k}: median sigma_G {np.median(sg[j]):.2f} vs sigma_bif {np.median(sb[j]):.2f}")
res = np.log(sb) - np.polyval(np.polyfit(np.log(sg), np.log(sb), 1), np.log(sg))
print(f"hierarchy BEYOND the Gaussian baseline (residual log sigma_bif): va {np.median(res[[i for i,p in enumerate(okg) if kind(p)=='va']]):+.2f}, vv {np.median(res[[i for i,p in enumerate(okg) if kind(p)=='vv']]):+.2f}, aa {np.median(res[[i for i,p in enumerate(okg) if kind(p)=='aa']]):+.2f}")

## E. Statistics: 1,000-resample intervals, effect sizes, class-level tests, sensitivity

In [ ]:
# 1,000 bootstrap resamples of images (with their noise draws) within each class, main coordinate, test set
NB = 1000; BOOT = {}; t0 = time.time()
for a, b in PAIRS:
    if not MAIN[(a, b)]: continue
    rows = pair_rows(YT4, a, b); Z = z_pair(L_MAIN, rows, a, b); ya = YT4[rows] == a; ia = np.where(ya)[0]; ib = np.where(~ya)[0]; hs, ss = [], []
    for _ in range(NB):
        sel = np.concatenate([rng.choice(ia, len(ia)), rng.choice(ib, len(ib))]); r = census_from_z([z[sel] for z in Z], ya[sel])
        if r: hs.append(r['h']); ss.append(r['sigma'])
    BOOT[(a, b)] = (np.array(hs), np.array(ss))
print(f"bootstrap done ({time.time()-t0:.0f}s)")
print(f"{'pair':>22} {'sigma_bif [95% CI]':>24} {'h [95% CI]':>28} {'equiv. split':>13} {'> class. null max':>18}")
H, HLO, HHI = [], [], []
for p in PAIRS:
    if p not in BOOT: print(f"{CLASSES[p[0]]+'/'+CLASSES[p[1]]:>22}  not bimodal"); continue
    hs, ss = BOOT[p]; hmed, hlo, hhi = np.median(hs), *np.percentile(hs, [2.5, 97.5]); smed, slo, shi = np.median(ss), *np.percentile(ss, [2.5, 97.5])
    H.append(hmed); HLO.append(hlo); HHI.append(hhi)
    above = (abs(hmed) > null_raw.max()) and (hlo > 0 or hhi < 0)
    print(f"{CLASSES[p[0]]+'/'+CLASSES[p[1]]:>22} {smed:8.2f} [{slo:5.2f},{shi:5.2f}] {hmed:+9.3f} [{hlo:+.3f},{hhi:+.3f}] {100*eq_imbalance(hmed):9.0f}/{100-100*eq_imbalance(hmed):.0f} {str(above):>18}")
H = np.array(H); print(f"median |h| {np.median(np.abs(H)):.3f} (equivalent split {100*eq_imbalance(np.median(np.abs(H))):.0f}/{100-100*eq_imbalance(np.median(np.abs(H))):.0f}); "
      f"pairs whose 95% CI excludes 0 AND |h| exceeds the maximum of the through-classifier null: {sum((abs(h) > null_raw.max()) and (l > 0 or u < 0) for h, l, u in zip(H, HLO, HHI))}/{len(H)}")
# class-level tests for the tighter-class rule and the hierarchy (each class appears in 9 pairs)
VAR = np.array([float(XTR[YTR == c].var(0).mean()) for c in range(10)]); okp = [p for p in PAIRS if p in BOOT]; hh = np.array([np.median(BOOT[p][0]) for p in okp])
def rule_stats(var): dv = np.array([var[a]-var[b] for a, b in okp]); return np.corrcoef(hh, dv)[0, 1], int(np.sum(np.sign(hh) == -np.sign(dv)))
r_obs, n_obs = rule_stats(VAR); perm_r, perm_n = [], []
for _ in range(10000): pv = rng.permutation(VAR); r_, n_ = rule_stats(pv); perm_r.append(r_); perm_n.append(n_)
perm_r, perm_n = np.array(perm_r), np.array(perm_n)
print(f"tighter-class rule: r = {r_obs:.3f} (class-permutation p = {np.mean(perm_r <= r_obs):.4f}); count {n_obs}/{len(okp)} (permutation p = {np.mean(perm_n >= n_obs):.4f})")
for c in range(10):
    sub = [i for i, p in enumerate(okp) if c not in p]; dv = np.array([VAR[a]-VAR[b] for a, b in okp]); print(f"   leave out {CLASSES[c]:>10}: r = {np.corrcoef(hh[sub], dv[sub])[0,1]:+.3f}, count {int(np.sum(np.sign(hh[sub]) == -np.sign(dv[sub])))}/{len(sub)}")
from itertools import combinations as comb
sb_all = {p: np.median(BOOT[p][1]) for p in okp}
def hier_ratio(veh): kd = lambda p: 'va' if ((p[0] in veh) != (p[1] in veh)) else ('vv' if p[0] in veh else 'aa'); va = [sb_all[p] for p in okp if kd(p) == 'va']; aa = [sb_all[p] for p in okp if kd(p) == 'aa']; return np.median(va)/np.median(aa)
obs = hier_ratio(VEH); allr = np.array([hier_ratio(set(v)) for v in comb(range(10), 4)])
print(f"hierarchy: median sigma_bif(vehicle/animal)/median sigma_bif(animal/animal) = {obs:.2f}; rank among all 210 choices of 4 'vehicle' classes: {int(np.sum(allr >= obs))}/210 (exact p = {np.mean(allr >= obs):.4f})")
# sensitivity of sigma_bif and h to bandwidth, merge/prominence thresholds and a persistence rule
print("sensitivity (vs default bw 1.5x Silverman, merge 0.03, prominence 0.05, persist 1):")
for lab, kw in [('bw 0.75x', dict(bwf=0.75)), ('bw 1.0x', dict(bwf=1.0)), ('bw 2.0x', dict(bwf=2.0)), ('merge 0.06, prom 0.10', dict(merge=0.06, prominence=0.10)), ('persist 2', dict(persist=2)), ('persist 3', dict(persist=3))]:
    hs_, ss_, h0, s0 = [], [], [], []
    for p in okp:
        rows = pair_rows(YT4, *p); r = census_from_z(z_pair(L_MAIN, rows, *p), YT4[rows] == p[0], **kw)
        if r: hs_.append(r['h']); ss_.append(r['sigma']); h0.append(MAIN[p]['h']); s0.append(MAIN[p]['sigma'])
    hs_, ss_, h0, s0 = map(np.array, (hs_, ss_, h0, s0))
    print(f"   {lab:>24}: n={len(hs_)}, corr h {np.corrcoef(h0, hs_)[0,1]:.3f}, median |dh| {np.median(np.abs(hs_-h0)):.4f}; corr log sigma {np.corrcoef(np.log(s0), np.log(ss_))[0,1]:.3f}, median ratio {np.median(ss_/s0):.2f}")
np.savez_compressed('/kaggle/working/step20_summary.npz', H=H, HLO=HLO, HHI=HHI, null_raw=null_raw, null_sym=null_sym, h30=h30)

## What to report back
- **A.** The 20 null lines; the three summary lines; the histogram.
- **B.** The calibration line (slope, nonlinearity, monotonicity) and the three raw-vs-variant lines.
- **C/D.** ResNet vs VGG; the Gaussian-baseline lines, including the "hierarchy beyond the baseline" line.
- **E.** The per-pair table, the effect-size summary, the permutation and leave-one-out lines, the hierarchy permutation line and the sensitivity table.
- Test accuracies by noise level (printed near the top).